# LABORATORIO 4 — DCGAN: Generación de Rostros de Anime
**Ingeniería en Ciencias de la Computación**  
**Dataset:** Anime Face Dataset (~63,000 imágenes, Kaggle)  
**Resolución de salida:** 64×64 píxeles RGB

## 1. Marco Teórico

### ¿Qué es una GAN?
Una **GAN** (*Generative Adversarial Network*) es una arquitectura de red neuronal propuesta por Ian Goodfellow en 2014. Está formada por dos redes que compiten entre sí durante el entrenamiento:

- **Generador (G):** Recibe un vector de ruido aleatorio $z$ (espacio latente) y genera imágenes sintéticas que intentan engañar al discriminador.
- **Discriminador (D):** Recibe imágenes (reales o generadas) y aprende a distinguirlas, devolviendo una probabilidad $P(real)$.

El objetivo del entrenamiento es un juego minimax:

$$\min_G \max_D \; \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

### ¿Qué es una DCGAN?
Una **DCGAN** (*Deep Convolutional GAN*) es una evolución de las GANs que reemplaza las capas densas por **capas convolucionales**, logrando resultados mucho más nítidos en síntesis de imágenes. Las mejoras clave son:

| Componente | Decisión de diseño |
|---|---|
| Generador | Convoluciones transpuestas para upsampling |
| Discriminador | Convoluciones con stride para downsampling |
| Activaciones | ReLU en G, LeakyReLU en D |
| Normalización | BatchNorm en ambas redes |
| Salida de G | Tanh (rango $[-1, 1]$) |
| Salida de D | Sigmoid (probabilidad $[0, 1]$) |

### Dataset: Anime Face Dataset
El dataset proviene de Kaggle (`splcher/animefacedataset`) y contiene aproximadamente **63,565 imágenes** de rostros de anime extraídas del sitio *www.getchu.com*. Las imágenes son variadas en estilo, color y expresión, lo que hace que la GAN deba aprender distribuciones complejas.

## 2. Instalación y Configuración

Antes de comenzar, se debe descargar el dataset desde Kaggle. Para ello se necesita tener configurado el archivo `kaggle.json` con las credenciales de la cuenta de Kaggle.

In [ ]:
# Instalar dependencias necesarias
# !pip install kaggle

# Descargar dataset desde Kaggle (ejecutar una sola vez)
# !kaggle datasets download -d splcher/animefacedataset
# !unzip animefacedataset.zip -d ./data/anime_faces

# Alternativamente, si ya tienes el dataset descargado:
# DATA_PATH = './data/anime_faces/images'

import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de entrenamiento: {device}')

## 3. Carga y Exploración del Dataset

Las imágenes del dataset están almacenadas en una carpeta plana (sin subclases). Se utiliza `torchvision.datasets.ImageFolder` después de crear una subcarpeta, o directamente con un `Dataset` personalizado. Dado que el dataset no tiene clases separadas, se usa `ImageFolder` con la ruta apuntando al directorio contenedor.

Se aplican las siguientes transformaciones:
- **Resize a 64×64:** Estandariza el tamaño de entrada.
- **CenterCrop:** Elimina bordes no deseados.
- **ToTensor:** Convierte a tensor $[0, 1]$.
- **Normalize a $[-1, 1]$:** Necesario para la activación `Tanh` del generador.

In [ ]:
# ─── Parámetros globales ───────────────────────────────────────────────────────
IMAGE_SIZE  = 64          # Resolución de salida: 64×64
BATCH_SIZE  = 128         # Tamaño del mini-batch
LATENT_DIM  = 100         # Dimensión del vector de ruido z
NUM_EPOCHS  = 50          # Épocas de entrenamiento
LR          = 0.0002      # Tasa de aprendizaje (recomendación DCGAN original)
BETA1       = 0.5         # Parámetro β1 del optimizador Adam
FEATURES_G  = 64          # Base de canales del generador
FEATURES_D  = 64          # Base de canales del discriminador
NC          = 3           # Canales de color (RGB)

# Ruta al dataset — ajustar según ubicación local
DATA_PATH = './data/anime_faces'

# ─── Transformaciones ──────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),   # Media por canal
                         (0.5, 0.5, 0.5)),   # Desviación estándar → rango [-1, 1]
])

# ─── Carga del dataset ─────────────────────────────────────────────────────────
# El dataset de Kaggle tiene la estructura: images/ (carpeta con todas las imágenes)
# ImageFolder requiere al menos una subcarpeta, así que apuntamos al directorio padre
dataset = torchvision.datasets.ImageFolder(root=DATA_PATH, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'Total de imágenes en el dataset : {len(dataset):,}')
print(f'Batches por época               : {len(dataloader)}')
print(f'Tamaño de un batch              : {BATCH_SIZE} imágenes de {NC}×{IMAGE_SIZE}×{IMAGE_SIZE}')

In [ ]:
# ─── Visualización de muestras del dataset ────────────────────────────────────
def mostrar_muestras(dataloader, n=16):
    """Muestra una grilla de imágenes reales del dataset."""
    imgs, _ = next(iter(dataloader))
    imgs = imgs[:n]
    # Desnormalizar: de [-1,1] a [0,1]
    imgs = imgs * 0.5 + 0.5
    grid = torchvision.utils.make_grid(imgs, nrow=4, padding=2)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title('Muestras reales del dataset — Anime Faces', fontsize=14)
    plt.tight_layout()
    plt.show()

mostrar_muestras(dataloader)

## 4. Arquitectura del Generador

El **generador** toma como entrada un vector de ruido $z \in \mathbb{R}^{100}$ con forma `(100, 1, 1)` y mediante capas de **convolución transpuesta** aumenta progresivamente la resolución espacial hasta obtener una imagen de `3×64×64`.

El flujo de tensores es el siguiente:

```
z: (N, 100, 1, 1)
   → ConvTranspose2d → (N, 512, 4, 4)   + BatchNorm + ReLU
   → ConvTranspose2d → (N, 256, 8, 8)   + BatchNorm + ReLU
   → ConvTranspose2d → (N, 128, 16, 16) + BatchNorm + ReLU
   → ConvTranspose2d → (N,  64, 32, 32) + BatchNorm + ReLU
   → ConvTranspose2d → (N,   3, 64, 64) + Tanh
```

La activación **Tanh** en la salida produce valores en $[-1, 1]$, que coincide con la normalización aplicada al dataset.

In [ ]:
class Generator(nn.Module):
    """
    Generador DCGAN para imágenes RGB de 64×64.

    Parámetros
    ----------
    latent_dim : int
        Dimensión del vector de ruido de entrada (espacio latente).
    img_channels : int
        Canales de la imagen de salida (3 para RGB).
    features_g : int
        Número base de filtros. Se multiplica en cada capa.
    """
    def __init__(self, latent_dim=100, img_channels=3, features_g=64):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            # Bloque 1: (N, latent_dim, 1, 1) → (N, 512, 4, 4)
            # kernel=4, stride=1, padding=0  →  H_out = (1-1)*1 - 0 + 4 = 4
            nn.ConvTranspose2d(latent_dim, features_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(inplace=True),

            # Bloque 2: (N, 512, 4, 4) → (N, 256, 8, 8)
            # kernel=4, stride=2, padding=1  →  H_out = (4-1)*2 - 2 + 4 = 8
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(inplace=True),

            # Bloque 3: (N, 256, 8, 8) → (N, 128, 16, 16)
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(inplace=True),

            # Bloque 4: (N, 128, 16, 16) → (N, 64, 32, 32)
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(inplace=True),

            # Capa de salida: (N, 64, 32, 32) → (N, 3, 64, 64)
            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # Salida en rango [-1, 1]
        )

    def forward(self, z):
        return self.model(z)


# Verificación de dimensiones
netG = Generator(latent_dim=LATENT_DIM, img_channels=NC, features_g=FEATURES_G).to(device)
z_test = torch.randn(4, LATENT_DIM, 1, 1).to(device)
out_test = netG(z_test)
print(f'Entrada del generador : {z_test.shape}')
print(f'Salida del generador  : {out_test.shape}  ← debe ser [4, 3, 64, 64]')
print(f'\nParámetros entrenables del generador: {sum(p.numel() for p in netG.parameters()):,}')

## 5. Arquitectura del Discriminador

El **discriminador** es una CNN clásica que toma imágenes de `3×64×64` y devuelve una probabilidad de que la imagen sea real. En cada capa se reduce la resolución a la mitad usando `stride=2`.

El flujo de tensores es el siguiente:

```
x: (N, 3, 64, 64)
   → Conv2d → (N,  64, 32, 32) + LeakyReLU
   → Conv2d → (N, 128, 16, 16) + BatchNorm + LeakyReLU
   → Conv2d → (N, 256,  8,  8) + BatchNorm + LeakyReLU
   → Conv2d → (N, 512,  4,  4) + BatchNorm + LeakyReLU
   → Conv2d → (N,   1,  1,  1) + Sigmoid   → probabilidad
```

Se usa **LeakyReLU** (con pendiente 0.2 para valores negativos) en lugar de ReLU para evitar el problema del gradiente nulo en el discriminador.

In [ ]:
class Discriminator(nn.Module):
    """
    Discriminador DCGAN para imágenes RGB de 64×64.

    Parámetros
    ----------
    img_channels : int
        Canales de la imagen de entrada (3 para RGB).
    features_d : int
        Número base de filtros. Se duplica en cada capa.
    """
    def __init__(self, img_channels=3, features_d=64):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            # Bloque 1: (N, 3, 64, 64) → (N, 64, 32, 32)   — sin BatchNorm en la 1ª capa
            nn.Conv2d(img_channels, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque 2: (N, 64, 32, 32) → (N, 128, 16, 16)
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque 3: (N, 128, 16, 16) → (N, 256, 8, 8)
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Bloque 4: (N, 256, 8, 8) → (N, 512, 4, 4)
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Capa de salida: (N, 512, 4, 4) → (N, 1, 1, 1) → escalar
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()  # Probabilidad P(real)
        )

    def forward(self, x):
        return self.model(x).view(-1)  # Aplanar a (N,)


# Verificación de dimensiones
netD = Discriminator(img_channels=NC, features_d=FEATURES_D).to(device)
x_test = torch.randn(4, NC, IMAGE_SIZE, IMAGE_SIZE).to(device)
out_test = netD(x_test)
print(f'Entrada del discriminador : {x_test.shape}')
print(f'Salida del discriminador  : {out_test.shape}  ← debe ser [4]')
print(f'\nParámetros entrenables del discriminador: {sum(p.numel() for p in netD.parameters()):,}')

## 6. Inicialización de Pesos

Siguiendo las recomendaciones del paper original de DCGAN (Radford et al., 2015), los pesos se inicializan con una **distribución normal** $\mathcal{N}(0, 0.02)$ para las capas convolucionales y lineales, y con $\mathcal{N}(1, 0.02)$ para los parámetros de escala de BatchNorm (con sesgo en 0).

Esta inicialización cuidadosa es crucial para la estabilidad del entrenamiento en GANs.

In [ ]:
def inicializar_pesos(modelo):
    """
    Inicializa los pesos de la red según las recomendaciones del paper DCGAN.
    - Capas Conv y ConvTranspose : N(0, 0.02)
    - Capas BatchNorm            : peso N(1, 0.02), sesgo 0
    """
    nombre_clase = modelo.__class__.__name__

    if 'Conv' in nombre_clase:                          # Conv2d y ConvTranspose2d
        nn.init.normal_(modelo.weight.data, 0.0, 0.02)
        if modelo.bias is not None:
            nn.init.constant_(modelo.bias.data, 0)

    elif 'BatchNorm' in nombre_clase:                   # BatchNorm2d
        nn.init.normal_(modelo.weight.data, 1.0, 0.02)  # Escala γ ≈ 1
        nn.init.constant_(modelo.bias.data, 0.0)         # Sesgo β = 0


# Aplicar inicialización a ambas redes
netG.apply(inicializar_pesos)
netD.apply(inicializar_pesos)
print('Pesos inicializados correctamente en G y D.')

## 7. Configuración del Entrenamiento

Se definen los **optimizadores** y la **función de pérdida**.

- **Función de pérdida:** Binary Cross-Entropy (`BCELoss`), que mide qué tan bien el discriminador clasifica imágenes reales vs. falsas.
- **Optimizador:** Adam con $lr=0.0002$ y $\beta_1=0.5$ para ambas redes (valor recomendado para GANs).
- **Ruido fijo:** Un vector de ruido fijo `z_fijo` se usa para visualizar el progreso del generador a lo largo de las épocas con las mismas semillas aleatorias.

In [ ]:
# Función de pérdida
criterion = nn.BCELoss()

# Optimizadores (Adam con β1=0.5 recomendado para GANs)
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))

# Ruido fijo para visualizar progreso (siempre el mismo)
z_fijo = torch.randn(16, LATENT_DIM, 1, 1, device=device)

# Etiquetas
LABEL_REAL = 1.0
LABEL_FALSO = 0.0

# Directorio para guardar checkpoints
os.makedirs('./checkpoints_anime', exist_ok=True)
os.makedirs('./imagenes_generadas', exist_ok=True)

print('Configuración de entrenamiento lista.')
print(f'  Épocas          : {NUM_EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Learning rate   : {LR}')
print(f'  Latent dim (z)  : {LATENT_DIM}')

## 8. Bucle de Entrenamiento

El entrenamiento sigue dos pasos por cada mini-batch:

**Paso 1 — Entrenar el Discriminador:**
1. Calcular la pérdida con imágenes **reales** (etiqueta = 1): $\mathcal{L}_D^{real} = \text{BCE}(D(x), 1)$
2. Calcular la pérdida con imágenes **falsas** (etiqueta = 0): $\mathcal{L}_D^{fake} = \text{BCE}(D(G(z)), 0)$
3. Actualizar D con la pérdida total: $\mathcal{L}_D = \mathcal{L}_D^{real} + \mathcal{L}_D^{fake}$

**Paso 2 — Entrenar el Generador:**
1. Pasar imágenes falsas por D
2. Calcular pérdida usando etiqueta = 1 (el generador quiere engañar a D): $\mathcal{L}_G = \text{BCE}(D(G(z)), 1)$
3. Actualizar G

In [ ]:
def guardar_imagenes(generador, z_fijo, epoch, carpeta='./imagenes_generadas'):
    """Genera y guarda una grilla de imágenes con z_fijo para comparar progreso."""
    generador.eval()
    with torch.no_grad():
        imgs_falsas = generador(z_fijo).cpu()
    generador.train()
    imgs_falsas = imgs_falsas * 0.5 + 0.5  # desnormalizar a [0,1]
    grid = torchvision.utils.make_grid(imgs_falsas, nrow=4, padding=2)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title(f'Imágenes generadas — Época {epoch}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{carpeta}/epoch_{epoch:03d}.png', dpi=100)
    plt.show()


def entrenar_dcgan(netG, netD, dataloader, num_epochs,
                   optimizerG, optimizerD, criterion, device,
                   z_fijo, checkpoint_dir='./checkpoints_anime'):
    """
    Bucle principal de entrenamiento de la DCGAN.

    Retorna
    -------
    hist : dict con listas 'G_loss' y 'D_loss' por época.
    """
    hist = {'G_loss': [], 'D_loss': []}

    for epoch in range(1, num_epochs + 1):
        g_losses_epoch = []
        d_losses_epoch = []

        for i, (imgs_reales, _) in enumerate(dataloader):
            imgs_reales = imgs_reales.to(device)
            batch_size  = imgs_reales.size(0)

            # ── PASO 1: Entrenar el Discriminador ──────────────────────────────
            netD.zero_grad()

            # Con imágenes reales
            etiquetas_reales = torch.full((batch_size,), LABEL_REAL,
                                          dtype=torch.float, device=device)
            salida_real = netD(imgs_reales)
            perdida_D_real = criterion(salida_real, etiquetas_reales)
            perdida_D_real.backward()

            # Con imágenes falsas
            z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            imgs_falsas = netG(z)
            etiquetas_falsas = torch.full((batch_size,), LABEL_FALSO,
                                           dtype=torch.float, device=device)
            salida_falsa = netD(imgs_falsas.detach())  # .detach() evita gradientes en G
            perdida_D_falsa = criterion(salida_falsa, etiquetas_falsas)
            perdida_D_falsa.backward()

            perdida_D = perdida_D_real + perdida_D_falsa
            optimizerD.step()

            # ── PASO 2: Entrenar el Generador ──────────────────────────────────
            netG.zero_grad()

            # El generador quiere que D clasifique sus imágenes como reales
            salida_falsa_para_G = netD(imgs_falsas)
            perdida_G = criterion(salida_falsa_para_G, etiquetas_reales)
            perdida_G.backward()
            optimizerG.step()

            # Acumular pérdidas del batch
            d_losses_epoch.append(perdida_D.item())
            g_losses_epoch.append(perdida_G.item())

            if i % 100 == 0:
                print(f'[Época {epoch:3d}/{num_epochs}] '
                      f'Paso {i:4d}/{len(dataloader)} '
                      f'| Pérdida D: {perdida_D.item():.4f} '
                      f'| Pérdida G: {perdida_G.item():.4f}')

        # Métricas por época
        hist['D_loss'].append(np.mean(d_losses_epoch))
        hist['G_loss'].append(np.mean(g_losses_epoch))

        # Visualizar y guardar progreso cada 5 épocas
        if epoch % 5 == 0 or epoch == 1:
            guardar_imagenes(netG, z_fijo, epoch)

        # Guardar checkpoint
        torch.save({
            'epoch': epoch,
            'generator_state_dict'    : netG.state_dict(),
            'discriminador_state_dict': netD.state_dict(),
            'optimizerG_state_dict'   : optimizerG.state_dict(),
            'optimizerD_state_dict'   : optimizerD.state_dict(),
        }, f'{checkpoint_dir}/checkpoint_epoch_{epoch:03d}.pth')

        print(f'✓ Checkpoint guardado — Época {epoch}/{num_epochs} '
              f'| D_loss media: {hist["D_loss"][-1]:.4f} '
              f'| G_loss media: {hist["G_loss"][-1]:.4f}')

    return hist


# ─── Ejecutar entrenamiento ────────────────────────────────────────────────────
hist = entrenar_dcgan(
    netG, netD, dataloader, NUM_EPOCHS,
    optimizerG, optimizerD, criterion, device, z_fijo
)

## 9. Análisis de las Pérdidas de Entrenamiento

En una GAN bien entrenada, se espera el siguiente comportamiento de las pérdidas:

- **Pérdida del Discriminador (D):** Tiende a estabilizarse alrededor de **0.5**, ya que en equilibrio no puede distinguir imágenes reales de falsas mejor que el azar.
- **Pérdida del Generador (G):** Generalmente oscila y puede crecer, ya que el discriminador también mejora continuamente.

> ⚠️ A diferencia de la clasificación tradicional donde buscamos que **todas** las pérdidas disminuyan, en GANs el objetivo es el **equilibrio de Nash** entre G y D, no la minimización simultánea.

In [ ]:
def graficar_perdidas(hist):
    """Visualiza las curvas de pérdida del generador y discriminador por época."""
    epocas = range(1, len(hist['G_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pérdida del generador
    axes[0].plot(epocas, hist['G_loss'], color='steelblue', linewidth=2, label='G Loss')
    axes[0].set_xlabel('Época', fontsize=12)
    axes[0].set_ylabel('Pérdida media', fontsize=12)
    axes[0].set_title('Pérdida del Generador', fontsize=13)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Pérdida del discriminador
    axes[1].plot(epocas, hist['D_loss'], color='tomato', linewidth=2, label='D Loss')
    axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Equilibrio (0.5)')
    axes[1].set_xlabel('Época', fontsize=12)
    axes[1].set_ylabel('Pérdida media', fontsize=12)
    axes[1].set_title('Pérdida del Discriminador', fontsize=13)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.suptitle('Curvas de entrenamiento — DCGAN Anime Faces', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('./imagenes_generadas/curvas_perdida.png', dpi=100, bbox_inches='tight')
    plt.show()

graficar_perdidas(hist)

## 10. Generación de Nuevas Imágenes

Una vez entrenado el modelo, el **discriminador se descarta** y solo se usa el **generador** para producir nuevas imágenes. Cada vector $z$ diferente produce un rostro de anime distinto.

In [ ]:
def generar_imagenes(generador, latent_dim, n=25, device='cuda', titulo='Imágenes generadas'):
    """
    Genera y muestra n imágenes nuevas a partir de vectores z aleatorios.

    Parámetros
    ----------
    generador  : modelo Generator entrenado
    latent_dim : dimensión del espacio latente
    n          : cantidad de imágenes a generar
    """
    generador.eval()
    with torch.no_grad():
        z = torch.randn(n, latent_dim, 1, 1, device=device)
        imgs = generador(z).cpu()

    imgs = imgs * 0.5 + 0.5  # desnormalizar a [0,1]
    filas = int(np.ceil(np.sqrt(n)))
    grid = torchvision.utils.make_grid(imgs, nrow=filas, padding=3, normalize=False)

    plt.figure(figsize=(12, 12))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()


# Generar 25 rostros de anime nuevos
generar_imagenes(netG, LATENT_DIM, n=25, device=device,
                 titulo='Rostros de Anime Generados por DCGAN')

## 11. Interpolación en el Espacio Latente

Una propiedad fascinante del espacio latente aprendido por el generador es que se puede **interpolar** entre dos vectores $z_1$ y $z_2$ y obtener una transición suave entre dos rostros de anime. Esto demuestra que el generador ha aprendido una representación continua y estructurada del espacio de imágenes.

In [ ]:
def interpolar_latente(generador, latent_dim, pasos=10, device='cuda'):
    """
    Interpola linealmente entre dos vectores z y muestra la transición.
    """
    generador.eval()
    z1 = torch.randn(1, latent_dim, 1, 1, device=device)
    z2 = torch.randn(1, latent_dim, 1, 1, device=device)

    alphas = torch.linspace(0, 1, pasos)
    imgs_interpoladas = []

    with torch.no_grad():
        for alpha in alphas:
            z_interp = (1 - alpha) * z1 + alpha * z2
            img = generador(z_interp).cpu()
            imgs_interpoladas.append(img)

    imgs_interpoladas = torch.cat(imgs_interpoladas, dim=0)
    imgs_interpoladas = imgs_interpoladas * 0.5 + 0.5

    grid = torchvision.utils.make_grid(imgs_interpoladas, nrow=pasos, padding=2)
    plt.figure(figsize=(20, 3))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title('Interpolación en el espacio latente — de z₁ a z₂', fontsize=13)
    plt.tight_layout()
    plt.show()


interpolar_latente(netG, LATENT_DIM, pasos=10, device=device)

## 12. Cargar Checkpoint y Continuar Entrenamiento

El sistema de checkpoints permite retomar el entrenamiento desde cualquier época sin perder el progreso acumulado.

In [ ]:
def cargar_checkpoint(ruta_checkpoint, netG, netD, optimizerG, optimizerD, device):
    """
    Carga un checkpoint guardado y restaura el estado de los modelos y optimizadores.
    """
    checkpoint = torch.load(ruta_checkpoint, map_location=device, weights_only=True)

    netG.load_state_dict(checkpoint['generator_state_dict'])
    netD.load_state_dict(checkpoint['discriminador_state_dict'])
    optimizerG.load_state_dict(checkpoint['optimizerG_state_dict'])
    optimizerD.load_state_dict(checkpoint['optimizerD_state_dict'])

    epoca_guardada = checkpoint['epoch']
    print(f'Checkpoint cargado correctamente — Época {epoca_guardada}')
    return epoca_guardada


# Ejemplo de uso:
# epoca_inicio = cargar_checkpoint(
#     './checkpoints_anime/checkpoint_epoch_050.pth',
#     netG, netD, optimizerG, optimizerD, device
# )
# Luego continuar entrenamiento desde epoca_inicio + 1

## 13. Resumen y Conclusiones

En este laboratorio se implementó una **DCGAN** (*Deep Convolutional Generative Adversarial Network*) capaz de generar rostros de anime de **64×64 píxeles** a partir de vectores de ruido aleatorio.

### Arquitectura utilizada

| Red | Capas | Parámetros |
|---|---|---|
| **Generador** | 5 ConvTranspose2d + BatchNorm + ReLU/Tanh | ~3.5M |
| **Discriminador** | 5 Conv2d + BatchNorm + LeakyReLU/Sigmoid | ~2.8M |

### Decisiones de diseño clave
- **Sin capas densas:** Las convolucionales capturan mejor la estructura espacial de las imágenes.
- **BatchNorm:** Estabiliza el entrenamiento evitando el colapso de modo.
- **LeakyReLU en D:** Evita gradientes nulos que bloquearían el aprendizaje del discriminador.
- **Adam con β₁=0.5:** Más apropiado que los valores por defecto para la dinámica adversarial.
- **Etiquetas suaves (opcional):** Usar 0.9 en lugar de 1.0 para las reales mejora la estabilidad.

### Comportamiento esperado de las pérdidas
- La pérdida del **Discriminador** converge hacia ~0.5 (equilibrio).
- La pérdida del **Generador** oscila y suele crecer moderadamente a medida que el discriminador mejora.

### Dataset
- **Nombre:** Anime Face Dataset
- **Fuente:** Kaggle (`splcher/animefacedataset`)
- **Tamaño:** ~63,565 imágenes RGB de rostros de anime
- **Resolución entrenada:** 64×64 píxeles
- **Validador del dataset:** Claros Herbas André Shaiel (CI: 67617441)